# Repaso avanzado — Online Retail II UCI
## Caso: Análisis estratégico para el equipo directivo

**Dataset**: Online Retail II (UCI Machine Learning Repository)
Transacciones reales de una tienda online del Reino Unido entre 2009 y 2011.

**Contexto**: El director de datos ha pedido un análisis completo antes de la reunión de estrategia del Q4.
Cada sección reproduce una petición real que llegaría por email o en una reunión.

---

Columnas del dataset:
- `Invoice` — número de factura (empieza con `C` si es cancelación/devolución)
- `StockCode` — código de producto
- `Description` — nombre del producto
- `Quantity` — unidades por línea (negativo en cancelaciones)
- `InvoiceDate` — fecha y hora de la transacción
- `Price` — precio unitario en GBP
- `Customer ID` — identificador de cliente (puede ser nulo)
- `Country` — país del cliente


## Setup — carga del dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def find_project_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / '.git').exists() or (parent / 'data').exists():
            return parent
    return current

PROJECT_ROOT = find_project_root()
DATA_EXTERNAL = PROJECT_ROOT / 'data' / 'external'

print(f"Check: Proyecto raíz: {PROJECT_ROOT}")
print(f"Check: Datos external: {DATA_EXTERNAL}")

Check: Proyecto raíz: C:\Users\alefe\OneDrive\Documentos\GitHub\data-analytics-project
Check: Datos external: C:\Users\alefe\OneDrive\Documentos\GitHub\data-analytics-project\data\external


In [2]:
df = pd.read_csv(DATA_EXTERNAL / "online_retail_II.csv")
print(f"Filas totales: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head(3)

Filas totales: 1,067,371 | Columnas: 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


---
## Ejercicio 1 — Diagnóstico inicial

**Orden**
> "Antes de tocar nada, necesito un diagnóstico completo del dataset.
> Cuántos nulos hay por columna, qué tipos tiene cada campo,
> cuántas filas son cancelaciones y cuántas tienen precio o cantidad negativa.
> Quiero saber exactamente con qué calidad de datos estamos trabajando."


In [3]:
def diagnostico_nulos(df):
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]

    # nulos es una Serie con el conteo por columna.
    # Dividir entre len(df) da proporcion 0-1, multiplicar por 100 la convierte
    # en porcentaje. .round(2) es metodo pandas: redondea toda la Serie de una vez.
    pct = (nulos / len(df) * 100).round(2)

    # pd.DataFrame({'col': serie}) construye una tabla a partir de un diccionario.
    # Como nulos y pct tienen el mismo indice (nombres de columna del df original),
    # pandas las alinea automaticamente. Buena practica: juntar metricas relacionadas
    # en una tabla en vez de imprimirlas por separado.
    tabla = pd.DataFrame({'nulos': nulos, 'pct_%': pct})
    print('[NULOS POR COLUMNA]')
    print(tabla)
    return tabla

In [4]:
def diagnostico_tipos(df):
    print('[TIPOS DE DATOS]')
    print(df.dtypes)
    return df.dtypes

In [5]:
def diagnostico_cancelaciones(df):
    cancelaciones = df[df['Invoice'].astype(str).str.startswith('C')]

    # len() sobre un DataFrame devuelve el numero de filas.
    # Se guarda en n para no llamar len(cancelaciones) dos veces seguidas
    # en la siguiente linea — buena practica de legibilidad.
    n = len(cancelaciones)

    # round() built-in de Python (no .round() de pandas) porque n ya es un
    # entero escalar, no una Serie. Sintaxis: round(valor, decimales).
    pct = round(n / len(df) * 100, 2)
    print(f'[CANCELACIONES]  {n:,} filas  ({pct}% del total)')
    return cancelaciones

In [6]:
def diagnostico_negativos(df):
    neg_qty   = df[df['Quantity'] <= 0]
    neg_price = df[df['Price']    <= 0]
    print('[NEGATIVOS]')

    # Dentro del f-string hay dos patrones:
    # {len(neg_qty):,}  -> los dos puntos abren el formato y la coma es
    #                      separador de miles: 1234567 se imprime como 1,234,567.
    #                      Buena practica en cualquier numero grande para humanos.
    # {round(...)}      -> calculo del porcentaje inline dentro del f-string.
    #                      Valido cuando el valor solo aparece una vez y no
    #                      merece su propia variable.
    print(f'  Quantity <= 0 : {len(neg_qty):,} filas ({round(len(neg_qty)/len(df)*100,2)}%)')
    print(f'  Price    <= 0 : {len(neg_price):,} filas ({round(len(neg_price)/len(df)*100,2)}%)')
    return neg_qty, neg_price

In [7]:
def ejecutar_diagnostico(df):
    # '=' * 45 es repeticion de string en Python: multiplicar un string por
    # un entero lo repite ese numero de veces. Produce 45 signos de igual.
    # Buena practica para separar visualmente secciones en el output de consola.
    print('=' * 60)
    print('DIAGNOSTICO INICIAL')
    print('=' * 60)
    diagnostico_nulos(df)
    print()
    diagnostico_tipos(df)
    print()
    diagnostico_cancelaciones(df)
    print()
    diagnostico_negativos(df)
    print('=' * 60)
    print('DIAGNOSTICO COMPLETADO')
    print('=' * 60)


ejecutar_diagnostico(df)

DIAGNOSTICO INICIAL


[NULOS POR COLUMNA]
              nulos  pct_%
Description    4382   0.41
Customer ID  243007  22.77

[TIPOS DE DATOS]
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object



[CANCELACIONES]  19,494 filas  (1.83% del total)

[NEGATIVOS]
  Quantity <= 0 : 22,950 filas (2.15%)
  Price    <= 0 : 6,207 filas (0.58%)
DIAGNOSTICO COMPLETADO


---
## Ejercicio 2 — Limpieza con criterio

**Email del director de datos (continuación):**
> "El año pasado el equipo anterior no limpió las cancelaciones y el revenue
> apareció inflado un 18% en el informe anual. No podemos repetir eso.
> Limpia el dataset con criterio y documenta cada decisión."

Aplica estas decisiones en orden y justifica cada una con un comentario:

1. Las filas con `Invoice` que empieza por `'C'` son devoluciones — decidir si eliminarlas o marcarlas
2. Las filas con `Quantity` <= 0 o `Price` <= 0 no representan ventas válidas
3. Las filas sin `Customer ID` no permiten análisis de cliente — decidir qué hacer con ellas
4. Eliminar duplicados exactos
5. Convertir `InvoiceDate` al tipo correcto
6. Verificar que después de la limpieza el shape tiene sentido respecto al original

Al terminar: imprime cuántas filas se perdieron en cada paso y el % sobre el total original.


In [8]:
# PASO 0 — Copiar el df antes de tocar nada.
# Si algo falla durante la limpieza, df sigue intacto para volver a empezar.
# .copy() hace una copia profunda: modificar df_clean no afecta a df.
df_clean = df.copy()
total_original = len(df_clean)


# PASO 1 — Marcar cancelaciones, NO eliminarlas todavia.
# Razon: el Ejercicio 5 necesita los productos mas devueltos.
# Si eliminamos estas filas aqui, perdemos esa informacion para siempre.
# La decision correcta es marcarlas y filtrar solo cuando hagamos ventas.

# .astype(str) — convierte Invoice a string por si hay valores mixtos
#               (a veces pandas lee numeros donde deberia haber texto)
# .str.startswith('C') — devuelve una Serie de True/False por cada fila
#                         True si la factura empieza por C, False si no
df_clean['es_cancelacion'] = df_clean['Invoice'].astype(str).str.startswith('C')

# .sum() sobre una Serie booleana cuenta los True (True=1, False=0)
n_cancel = df_clean['es_cancelacion'].sum()
pct      = round(n_cancel / total_original * 100, 2)

print(f'[CANCELACIONES]  {n_cancel:,} filas marcadas ({pct}% del total)')
print(f'Shape actual:    {df_clean.shape}  <- una columna mas, mismas filas')

[CANCELACIONES]  19,494 filas marcadas (1.83% del total)
Shape actual:    (1067371, 9)  <- una columna mas, mismas filas


---
## Ejercicio 3 — Columna de revenue y métricas base

**Email del CFO:**
> "Necesito una columna de revenue por línea de pedido para todos los cálculos
> posteriores. También quiero el revenue total, el número de transacciones únicas
> y el ticket medio por pedido. Solo con datos limpios."

- Crea la columna `Revenue` = Quantity × Price
- Calcula las tres métricas globales que pide el CFO
- ¿Cuál es el pedido con mayor revenue? ¿Y el producto con mayor revenue acumulado?


---
## Ejercicio 4 — Análisis por país

**Email del director comercial:**
> "¿En qué mercados estamos ganando dinero de verdad? Necesito ver
> por país: revenue total, número de pedidos únicos, número de clientes únicos
> y ticket medio por pedido. Ordénalo por revenue y dime si hay países
> con un ticket medio anormalmente alto o bajo respecto a la media global."

- Agrupa por `Country` usando `.agg()` con nombres descriptivos
- Añade una columna que muestre el % de revenue que representa cada país sobre el total
- Identifica qué porcentaje del revenue total acumulan los 3 primeros países (`cumsum`)
- Señala los países con ticket medio más del doble o menos de la mitad de la media global


In [9]:
# Tu análisis aquí


---
## Ejercicio 5 — Productos: rendimiento y devoluciones

**Email del director de producto:**
> "Quiero dos listas. Primero: los 10 productos que más revenue generan,
> con su descripción legible, no el código. Segundo: los 10 productos
> que más aparecen en cancelaciones (las filas con C que guardaste o marcaste).
> Si un producto está en las dos listas a la vez, quiero saberlo —
> eso es una señal de problema de calidad."

- Top 10 productos por revenue (con `Description`, no `StockCode`)
- Top 10 productos más devueltos
- Cruce entre ambas listas: ¿alguno aparece en los dos?


In [10]:
# Tu análisis aquí


---
## Ejercicio 6 — Evolución temporal mensual

**Email del CFO:**
> "El sistema de reporting dice que en noviembre de 2010 hubo una caída
> fuerte. Necesito la evolución mensual de revenue, número de pedidos
> y clientes activos. Con variación porcentual mes a mes para ver si
> la caída es real o un artefacto del dato."

- Agrupa por mes (`InvoiceDate` → periodo mensual)
- Calcula revenue mensual, pedidos únicos y clientes únicos
- Añade la variación porcentual respecto al mes anterior (`pct_change`)
- ¿La caída de noviembre 2010 es real? ¿Cuándo fue el mes con mayor revenue?


In [11]:
# Tu análisis aquí


---
## Ejercicio 7 — Merge con tabla de mercados

**Email del director de estrategia:**
> "He clasificado los países en mercados estratégicos. Cruza ese dato
> con tu análisis para ver si nuestro revenue está concentrado en
> los mercados correctos o estamos descuidando los prioritarios."

Tienes esta tabla auxiliar de mercados (cópiala y ejecuta):


In [12]:
mercados = pd.DataFrame({
    "Country": [
        "United Kingdom", "Germany", "France", "EIRE", "Spain",
        "Netherlands", "Belgium", "Switzerland", "Portugal", "Australia",
        "Norway", "Italy", "Channel Islands", "Finland", "Cyprus"
    ],
    "Mercado": [
        "Core", "Strategic", "Strategic", "Core", "Emerging",
        "Strategic", "Emerging", "Strategic", "Emerging", "Emerging",
        "Strategic", "Emerging", "Core", "Emerging", "Emerging"
    ],
    "Prioridad": [
        1, 2, 2, 1, 3,
        2, 3, 2, 3, 3,
        2, 3, 1, 3, 3
    ]
})
mercados.head()

,Country,Mercado,Prioridad
0,United Kingdom,Core,1
1,Germany,Strategic,2
2,France,Strategic,2
3,EIRE,Core,1
4,Spain,Emerging,3


- Une el DataFrame de resumen por país (del Ejercicio 4) con `mercados`
- Usa el tipo de join correcto para no perder países que no están en `mercados`
- Verifica la integridad: ¿cuántos países del dataset NO están en la tabla de mercados?
- ¿Cuánto revenue representa cada tipo de Mercado (Core / Strategic / Emerging)?
- ¿El revenue de Prioridad 1 justifica esa clasificación?


In [13]:
# Tu análisis aquí


---
## Ejercicio 8 — Outliers en transacciones

**Email del equipo de operaciones:**
> "Estamos revisando pedidos sospechosos. Hay cantidades que parecen
> errores de sistema (alguien tecleó de más) y precios que no tienen
> sentido para el catálogo normal. Usa criterio estadístico para identificarlos,
> no un umbral arbitrario."

- Detecta outliers en `Quantity` con el método IQR (×1.5)
- Detecta outliers en `Price` con z-score (umbral: 3 desviaciones)
- Para cada método: cuántos registros son outliers y qué % del dataset representan
- ¿Hay productos que aparecen consistentemente en outliers de cantidad?
- ¿Eliminarlos cambia significativamente el revenue total? Calcula la diferencia.


In [14]:
# Tu análisis aquí


---
## Ejercicio 9 — Perfil de cliente y correlación

**Email del equipo de CRM:**
> "Antes de hacer la segmentación necesito entender qué métricas de cliente
> están relacionadas entre sí. ¿Los clientes que compran más frecuente
> también gastan más? ¿O son clientes distintos?"

- Construye una tabla a nivel de cliente (`Customer ID`) con estas columnas:
  - `num_pedidos`: pedidos únicos
  - `total_revenue`: revenue acumulado
  - `ticket_medio`: revenue / num_pedidos
  - `num_productos`: productos distintos comprados
  - `dias_activo`: diferencia entre primera y última compra en días
- Calcula la matriz de correlación entre esas métricas
- Visualízala con un heatmap (`seaborn.heatmap`, `annot=True`)
- ¿Qué par de métricas tiene la correlación más alta? ¿Y la más baja?


In [15]:
# Tu análisis aquí


---
## Ejercicio 10 — Segmentación RFM

**Email del director de marketing:**
> "Necesito segmentos accionables, no solo números. Dime quiénes son
> mis mejores clientes, quiénes están en riesgo de irse, quiénes acaban
> de llegar y quiénes ya se fueron. Con eso diseño las campañas del Q4."

Fecha de referencia para calcular Recency: `2011-12-10` (último día del dataset).

- Calcula para cada cliente:
  - **Recency**: días desde su última compra hasta la fecha de referencia
  - **Frequency**: número de pedidos únicos
  - **Monetary**: revenue total
- Convierte cada métrica en un score del 1 al 4 usando `qcut` (cuartiles)
  — Recency: score 4 = compró más recientemente (invertir el orden)
- Crea un `RFM_Score` concatenando los tres scores como string (`"444"`, `"123"`, etc.)
- Clasifica cada cliente en un segmento según esta tabla:

| Segmento | Criterio |
|----------|----------|
| Campeón | R=4, F≥3, M≥3 |
| Leal | F≥3, M≥3 (cualquier R) |
| En riesgo | R≤2, F≥3 |
| Perdido | R=1, F≤2 |
| Nuevo | R=4, F=1 |
| Potencial | R≥3, F=2 |

- ¿Cuántos clientes hay en cada segmento?
- ¿Cuánto revenue acumulado representa cada segmento?
- ¿Qué segmento tiene el ticket medio más alto?


In [16]:
# Tu análisis aquí


---
## Preguntas de interpretación

Traduce los resultados numéricos a insights de negocio siguiendo la cadena:
**Observación → Patrón → Interpretación → Implicación**

Responde en texto (en una celda Markdown), no en código.

---

**1.** El Reino Unido representa más del 80% del revenue total. ¿Eso es una fortaleza o una vulnerabilidad para el negocio? Argumenta.

**2.** Mira los meses de noviembre y diciembre de ambos años. ¿Qué patrón ves? ¿Qué implicación tiene para la planificación de inventario y personal?

**3.** Si los clientes "En riesgo" tienen frecuencia alta pero recency baja, ¿qué dice eso sobre su comportamiento? ¿Qué campaña diseñarías para ellos?

**4.** Si la correlación entre `num_pedidos` y `total_revenue` es alta pero la correlación entre `ticket_medio` y `num_pedidos` es baja o negativa, ¿qué perfil de cliente tienen los mejores compradores?

**5.** Un producto aparece en el top 10 de revenue Y en el top 10 de devoluciones. ¿Qué hipótesis de negocio generarías? ¿Qué dato adicional necesitarías para confirmarla?


*Escribe tus respuestas aquí (doble clic para editar)*